In [1]:
# Importing all tools needed for training

import pandas as pd
import numpy as np
import boto3
import io
import json
import joblib
import warnings
warnings.filterwarnings('ignore')

import xgboost as xgb
import optuna
from optuna.samplers import TPESampler

import mlflow
import mlflow.xgboost
from mlflow.models.signature import infer_signature

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)

print("✅ All libraries imported!")

✅ All libraries imported!


In [2]:
# Loading our feature engineered data from S3

BUCKET_NAME = "churn-mlops-project-cherry"
s3 = boto3.client('s3')

def load_from_s3(bucket, key):
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(io.BytesIO(obj['Body'].read()))

# Load all 4 files we saved in Step 2
X_train = load_from_s3(BUCKET_NAME, 'data/features/X_train.csv')
X_test  = load_from_s3(BUCKET_NAME, 'data/features/X_test.csv')
y_train = load_from_s3(BUCKET_NAME, 'data/features/y_train.csv').squeeze()
y_test  = load_from_s3(BUCKET_NAME, 'data/features/y_test.csv').squeeze()

print("✅ Data loaded from S3!")
print(f"📊 X_train shape: {X_train.shape}")
print(f"📊 X_test shape:  {X_test.shape}")
print(f"📊 y_train shape: {y_train.shape}")
print(f"📊 y_test shape:  {y_test.shape}")

✅ Data loaded from S3!
📊 X_train shape: (911, 32)
📊 X_test shape:  (228, 32)
📊 y_train shape: (911,)
📊 y_test shape:  (228,)


In [4]:
# MLflow tracks every experiment we run
# We use LOCAL storage for tracking (SageMaker's own disk)
# and S3 only for storing the final model files

import os

# Create a local folder for MLflow to store experiment data
mlflow_dir = os.path.expanduser("~/churn-mlops/mlflow_tracking")
os.makedirs(mlflow_dir, exist_ok=True)

# Tell MLflow to use local folder - this fixes the S3 error
mlflow.set_tracking_uri(f"file://{mlflow_dir}")

# Create our experiment
mlflow.set_experiment("netflix-churn-prediction")

print("✅ MLflow configured successfully!")
print(f"📁 Tracking data saved at: {mlflow_dir}")
print(f"🧪 Experiment: netflix-churn-prediction")

2026/04/13 08:10:35 INFO mlflow.tracking.fluent: Experiment with name 'netflix-churn-prediction' does not exist. Creating a new experiment.


✅ MLflow configured successfully!
📁 Tracking data saved at: /home/sagemaker-user/churn-mlops/mlflow_tracking
🧪 Experiment: netflix-churn-prediction


In [5]:
# This is the heart of our training
# Optuna will call this function 50 times
# Each time with different settings (hyperparameters)
# It learns which settings work best - like a smart search

def objective(trial):
    # Optuna suggests different values each time it runs
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 500),
        'max_depth':         trial.suggest_int('max_depth', 3, 9),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight':  trial.suggest_int('min_child_weight', 1, 7),
        'gamma':             trial.suggest_float('gamma', 0, 1),
        'reg_alpha':         trial.suggest_float('reg_alpha', 0, 1),
        'reg_lambda':        trial.suggest_float('reg_lambda', 0, 1),
        'scale_pos_weight':  trial.suggest_float('scale_pos_weight', 0.5, 2.0),
        'use_label_encoder': False,
        'eval_metric':       'auc',
        'random_state':      42,
        'n_jobs':            -1
    }

    # Train model with these settings
    model = xgb.XGBClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_pred_proba)

    return auc

print("✅ Optuna objective function defined!")
print("🔍 Optuna will try 50 different combinations")
print("🎯 Goal: maximize AUC score")

✅ Optuna objective function defined!
🔍 Optuna will try 50 different combinations
🎯 Goal: maximize AUC score


In [6]:


print("🚀 Starting Optuna hyperparameter search...")
print("⏳ This will take 3-5 minutes. Please wait...\n")

# Create study - this is Optuna's word for an experiment
study = optuna.create_study(
    direction='maximize',    # we want to MAXIMIZE AUC
    sampler=TPESampler(seed=42)  # smart search algorithm
)

# Run 50 trials
study.optimize(
    objective,
    n_trials=50,
    show_progress_bar=True
)

print(f"\n✅ Optuna search complete!")
print(f"🏆 Best AUC score: {study.best_value:.4f}")
print(f"⚙️  Best settings found:")
for key, value in study.best_params.items():
    print(f"   {key}: {value}")

[I 2026-04-13 08:13:45,212] A new study created in memory with name: no-name-b695917e-d365-4bee-acec-d0f7a8098fd8


🚀 Starting Optuna hyperparameter search...
⏳ This will take 3-5 minutes. Please wait...



  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-04-13 08:13:45,539] Trial 0 finished with value: 0.994228549442093 and parameters: {'n_estimators': 250, 'max_depth': 9, 'learning_rate': 0.22227824312530747, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'min_child_weight': 2, 'gamma': 0.05808361216819946, 'reg_alpha': 0.8661761457749352, 'reg_lambda': 0.6011150117432088, 'scale_pos_weight': 1.5621088666940683}. Best is trial 0 with value: 0.994228549442093.
[I 2026-04-13 08:13:45,679] Trial 1 finished with value: 0.9930742593305117 and parameters: {'n_estimators': 108, 'max_depth': 9, 'learning_rate': 0.2514083658321223, 'subsample': 0.6849356442713105, 'colsample_bytree': 0.6727299868828402, 'min_child_weight': 2, 'gamma': 0.3042422429595377, 'reg_alpha': 0.5247564316322378, 'reg_lambda': 0.43194501864211576, 'scale_pos_weight': 0.9368437102970628}. Best is trial 0 with value: 0.994228549442093.
[I 2026-04-13 08:13:45,955] Trial 2 finished with value: 0.9932281646787227 and parameters: {'n_estimato

In [7]:


print("🚀 Training final model with best settings...")

with mlflow.start_run(run_name="xgboost-optuna-best"):

    # Use the best parameters Optuna found
    best_params = study.best_params
    best_params['use_label_encoder'] = False
    best_params['eval_metric'] = 'auc'
    best_params['random_state'] = 42
    best_params['n_jobs'] = -1

    # Train the final model
    final_model = xgb.XGBClassifier(**best_params)
    final_model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )

    # Make predictions
    y_pred       = final_model.predict(X_test)
    y_pred_proba = final_model.predict_proba(X_test)[:, 1]

    # Calculate all metrics
    accuracy  = accuracy_score(y_test, y_pred)
    auc       = roc_auc_score(y_test, y_pred_proba)
    f1        = f1_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall    = recall_score(y_test, y_pred)

    # Log parameters to MLflow
    mlflow.log_params(best_params)

    # Log metrics to MLflow
    mlflow.log_metric("accuracy",  accuracy)
    mlflow.log_metric("auc",       auc)
    mlflow.log_metric("f1_score",  f1)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall",    recall)

    # Log the model itself to MLflow
    signature = infer_signature(X_train, y_pred)
    mlflow.xgboost.log_model(
        final_model,
        "model",
        signature=signature
    )

    # Get the run ID - we need this later
    run_id = mlflow.active_run().info.run_id

print(f"\n✅ Final model trained and logged to MLflow!")
print(f"\n📊 Model Performance:")
print(f"   AUC Score:  {auc:.4f}  (higher is better, 1.0 = perfect)")
print(f"   Accuracy:   {accuracy:.4f}")
print(f"   F1 Score:   {f1:.4f}")
print(f"   Precision:  {precision:.4f}")
print(f"   Recall:     {recall:.4f}")
print(f"\n🔑 MLflow Run ID: {run_id}")

🚀 Training final model with best settings...

✅ Final model trained and logged to MLflow!

📊 Model Performance:
   AUC Score:  0.9966  (higher is better, 1.0 = perfect)
   Accuracy:   0.9561
   F1 Score:   0.9545
   Precision:  0.9813
   Recall:     0.9292

🔑 MLflow Run ID: f14098d1bcf54ddd8068fe0a5c61c675


In [8]:
# Let's see exactly how our model performs
# Classification report shows results for each class

print("📊 Detailed Classification Report:")
print("="*50)
print(classification_report(
    y_test, y_pred,
    target_names=['Stayed', 'Churned']
))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("📊 Confusion Matrix:")
print("="*50)
print(f"                 Predicted")
print(f"                 Stayed  Churned")
print(f"Actual Stayed  [{cm[0][0]:^6}  {cm[0][1]:^7}]")
print(f"Actual Churned [{cm[1][0]:^6}  {cm[1][1]:^7}]")
print(f"\n✅ Correctly identified {cm[0][0]} customers who stayed")
print(f"✅ Correctly identified {cm[1][1]} customers who churned")
print(f"❌ Missed {cm[1][0]} churners (said they would stay but they left)")

📊 Detailed Classification Report:
              precision    recall  f1-score   support

      Stayed       0.93      0.98      0.96       115
     Churned       0.98      0.93      0.95       113

    accuracy                           0.96       228
   macro avg       0.96      0.96      0.96       228
weighted avg       0.96      0.96      0.96       228

📊 Confusion Matrix:
                 Predicted
                 Stayed  Churned
Actual Stayed  [ 113       2   ]
Actual Churned [  8       105  ]

✅ Correctly identified 113 customers who stayed
✅ Correctly identified 105 customers who churned
❌ Missed 8 churners (said they would stay but they left)


In [9]:
# Save the model to S3 with proper versioning
# This is how production systems store models

import datetime

# Create a version number based on today's date
version = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# Save model locally first
local_model_path = '/tmp/xgboost_churn_model.pkl'
joblib.dump(final_model, local_model_path)

# Upload to S3 with version in the path
model_s3_key = f'models/v_{version}/xgboost_churn_model.pkl'
s3.upload_file(local_model_path, BUCKET_NAME, model_s3_key)

# Also save as 'latest' so the API always knows which one to use
s3.upload_file(local_model_path, BUCKET_NAME, 'models/latest/xgboost_churn_model.pkl')

# Save model metadata
metadata = {
    'version':    version,
    'run_id':     run_id,
    'auc':        round(auc, 4),
    'accuracy':   round(accuracy, 4),
    'f1_score':   round(f1, 4),
    'precision':  round(precision, 4),
    'recall':     round(recall, 4),
    'features':   X_train.columns.tolist(),
    'trained_at': str(datetime.datetime.now()),
    'model_key':  model_s3_key
}

s3.put_object(
    Bucket=BUCKET_NAME,
    Key='models/latest/metadata.json',
    Body=json.dumps(metadata, indent=2)
)

print(f"✅ Model saved to S3!")
print(f"📁 Versioned: s3://{BUCKET_NAME}/{model_s3_key}")
print(f"📁 Latest:    s3://{BUCKET_NAME}/models/latest/xgboost_churn_model.pkl")
print(f"\n🎉 Step 3 Complete! Your model is trained and saved!")
print(f"\n📊 Your model summary:")
print(f"   Version:  {version}")
print(f"   AUC:      {auc:.4f}")
print(f"   Accuracy: {accuracy:.4f}")

✅ Model saved to S3!
📁 Versioned: s3://churn-mlops-project-cherry/models/v_20260413_081716/xgboost_churn_model.pkl
📁 Latest:    s3://churn-mlops-project-cherry/models/latest/xgboost_churn_model.pkl

🎉 Step 3 Complete! Your model is trained and saved!

📊 Your model summary:
   Version:  20260413_081716
   AUC:      0.9966
   Accuracy: 0.9561
